# create crops from training images

In [3]:
import os
import json
from PIL import Image
import shutil

# Paths
images_folder = "dataset/images/train"
json_folder = "annotations/train"
crops_folder = "crops_per_label"

os.makedirs(crops_folder, exist_ok=True)

# Function to crop objects based on JSON annotations
def crop_objects_from_image(image_path, json_path, save_root):
    with open(json_path, "r") as f:
        data = json.load(f)

    image = Image.open(image_path)
    base_name = os.path.splitext(os.path.basename(image_path))[0]

    for obj in data.get("objects", []):
        label = obj["label"]
        bbox = obj["bbox"]
        xmin = int(bbox["xmin"])
        ymin = int(bbox["ymin"])
        xmax = int(bbox["xmax"])
        ymax = int(bbox["ymax"])

        cropped = image.crop((xmin, ymin, xmax, ymax))

        label_folder = os.path.join(save_root, label)
        os.makedirs(label_folder, exist_ok=True)

        crop_filename = f"{base_name}_{obj['key']}.png"
        crop_path = os.path.join(label_folder, crop_filename)
        cropped.save(crop_path)
        print(f"Saved crop: {crop_path}")

# Match images and JSON files and crop
image_files = [f for f in os.listdir(images_folder) if os.path.isfile(os.path.join(images_folder, f))]
json_files = {os.path.splitext(f)[0]: f for f in os.listdir(json_folder) if f.lower().endswith(".json")}

for img_file in image_files:
    img_name = os.path.splitext(img_file)[0]
    if img_name in json_files:
        img_path = os.path.join(images_folder, img_file)
        json_path = os.path.join(json_folder, json_files[img_name])
        crop_objects_from_image(img_path, json_path, crops_folder)

print("All crops done!")


Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_tr64yft61b73dudyjix53j.png
Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_df6ax8buw8ymu7iki2crt1.png
Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_zsmon3y9mlwtupmaulih0d.png
Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_g78zslajz2bow2w1b09yns.png
Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_r555atu3lfullp3uwdxk2o.png
Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_u395gf8qb8r4rwgu0j8rsi.png
Saved crop: crops_per_label\regulatory--keep-right--g4\--vWKSR3Rh8quTfK4AuKOQ_c43y1orc8tlkoibr9etbg3.png
Saved crop: crops_per_label\other-sign\--vWKSR3Rh8quTfK4AuKOQ_nkw17k5zhvf3himt3l9w0e.png
Saved crop: crops_per_label\other-sign\-0BXQnPOeVpklWx23Caxaw_4rbpnt2rqm4cmg3o5qe0tm.png
Saved crop: crops_per_label\other-sign\-0BXQnPOeVpklWx23Caxaw_aecmgvbqku8wl3ial3y8hl.png
Saved crop: crops_per_label\other-sign\-0BXQnPOeVpklWx23Caxaw_rix929d5s0d9zix0qicznd.png
Saved

# train model

In [10]:
import torch
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

# Paths
dataset_path = "crops_per_label"  # crops folder from above

# Data transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset
dataset = datasets.ImageFolder(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Load pretrained model
model = models.efficientnet_b0(pretrained=True)

# Replace classifier
num_classes = len(dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Simple training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(dataloader):.4f}")

print("Training done!")

Epoch 1/5, Loss: 3.5697
Epoch 2/5, Loss: 2.1030
Epoch 3/5, Loss: 1.7416
Epoch 4/5, Loss: 1.4800
Epoch 5/5, Loss: 1.2563
Training done!


In [11]:
torch.save(model.state_dict(), "street_sign_classifier_weights.pth")

In [12]:
# Load later
# Make sure you redefine the model architecture first
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.datasets import ImageFolder

train_dir = "crops_per_label"

dataset = ImageFolder(train_dir)
class_names = dataset.classes   # SORTED and consistent
num_classes = len(class_names)

print("Number of classes:", num_classes)
#print("Classes:", class_names)

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)

model.load_state_dict(torch.load("street_sign_classifier_weights.pth"))
model.eval()  # set to evaluation mode

Number of classes: 291


C:\Users\ralfm\AppData\Local\Temp\ipykernel_23276\4080485563.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("street_sign_classifier_we

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat